# AIRF SLM tool calling — Colab (Gemma only, target ≥ 93%)

**Models allowed here:** Gemma 3 270M / 1B / 4B and FunctionGemma 270M (≤ 4B).

Colab secrets:
- `HF_TOKEN` — must be allowed for this notebook (the toggle next to the secret)
- You must click **Agree and access** on the Gemma page while logged into the **same** Hugging Face account as the token

https://huggingface.co/google/gemma-3-270m-it


In [ ]:
import importlib.util, subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "torchao", "peft", "accelerate", "transformers", "huggingface_hub"])
print("ok")


In [ ]:
import json, math, os, random, re, time, urllib.request, zipfile, gc
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 0
random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else torch.float32
print("device", DEVICE, DTYPE)

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as e:
    print("colab userdata:", e)
    HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")

if HF_TOKEN:
    HF_TOKEN = str(HF_TOKEN).strip().strip('"').strip("'")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN introuvable. Colab → 🔑 Secrets → HF_TOKEN + interrupteur Notebook access ON. "
        "Runtime → Restart session."
    )
if not HF_TOKEN.startswith("hf_"):
    raise RuntimeError("Le secret HF_TOKEN ne ressemble pas à un token Hugging Face (doit commencer par hf_).")

from huggingface_hub import login, whoami, HfApi
login(token=HF_TOKEN, add_to_git_credential=False)
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

info = whoami(token=HF_TOKEN)
print("HF account:", info.get("name"), "| type:", info.get("auth", {}).get("type") or info.get("type"))
print("Si 403 Gemma: ouvre https://huggingface.co/google/gemma-3-270m-it CONNECTÉ EN TANT QUE", info.get("name"))
print("puis clique Agree and access. Attends 1–2 min, Runtime → Restart, relance.")

api = HfApi(token=HF_TOKEN)
try:
    api.model_info("google/gemma-3-270m-it")
    print("Gemma 3 270M: ACCESS OK")
except Exception as e:
    print("Gemma 3 270M: NO ACCESS ->", type(e).__name__, str(e)[:400])
    raise RuntimeError(
        "Ce token n'a PAS le droit de télécharger Gemma. "
        "Connecte-toi sur Hugging Face avec le compte affiché ci-dessus, "
        "accepte la licence sur https://huggingface.co/google/gemma-3-270m-it "
        "puis recrée un token Read (https://huggingface.co/settings/tokens) et mets-le dans HF_TOKEN."
    ) from e


In [ ]:
REPO = "https://raw.githubusercontent.com/aims-ai-research-foundations/airf-slm-tool-calling-hackathon/main/data"
DATA_DIR = Path("data"); DATA_DIR.mkdir(exist_ok=True)
for name in ("tools.json", "train.jsonl", "dev.jsonl"):
    urllib.request.urlretrieve(f"{REPO}/{name}", DATA_DIR / name)

def read_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text(encoding="utf-8").splitlines() if l.strip()]

def answer_of(row):
    return row["tool_calls"] if "tool_calls" in row else {"tool": row["tool"], "parameters": row["parameters"]}

def calls_of(answer):
    return answer if isinstance(answer, list) else [answer]

tools = {t["name"]: t for t in json.loads((DATA_DIR / "tools.json").read_text())}
train = [dict(row, answer=answer_of(row)) for row in read_jsonl(DATA_DIR / "train.jsonl")]
dev = read_jsonl(DATA_DIR / "dev.jsonl")
print(len(tools), "tools", len(train), "train", len(dev), "dev")


In [ ]:
# T4-safe: official Gemma 3 270M only (same as starter). Do not load 1B/4B on a T4 with this loop.
MODEL_NAME = "google/gemma-3-270m-it"
EPOCHS = 2
LORA_R = 16
LR = 2e-4
BATCH_TOKENS = 2048
INSTRUCTION = (
    "Reply with a JSON list of the tool calls for the request, with null for values not given.\nTools:\n"
)

print("loading tokenizer", MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def build_prompt(example):
    signatures = [f"{name}({', '.join(f'{p}: {t}' for p, t in tools[name]['parameters'].items())})" for name in example["tools"]]
    messages = [{"role": "system", "content": INSTRUCTION + "\n".join(signatures)}, {"role": "user", "content": example["request"]}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def target_text(answer):
    return json.dumps(calls_of(answer), ensure_ascii=False) + "<end_of_turn>"

by_category = defaultdict(list)
for ex in train:
    by_category[ex["meta"]["category"]].append(ex)
validation = [ex for group in by_category.values() for ex in random.Random(SEED).sample(group, 100)]
held = {ex["id"] for ex in validation}
train_pool = [ex for ex in train if ex["id"] not in held]
print(MODEL_NAME, "train_pool", len(train_pool), "val", len(validation))
print("cuda mem allocated GB", round(torch.cuda.memory_allocated()/1e9, 2) if torch.cuda.is_available() else 0)


In [ ]:
def normalise(value):
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return float(value)
    if isinstance(value, str):
        return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", "", value.lower())).strip()
    return value

def canonical(answer):
    try:
        return tuple((c["tool"], tuple(sorted((k, normalise(v)) for k, v in c["parameters"].items() if v is not None))) for c in calls_of(answer))
    except (TypeError, KeyError, AttributeError):
        return None

def score(predictions):
    hits = defaultdict(list)
    for ex in validation:
        hits[ex["meta"]["category"]].append(canonical(predictions.get(ex["id"])) == canonical(ex["answer"]))
    return {"overall": float(np.mean(sum(hits.values(), [])))} | {k: float(np.mean(hits[k])) for k in ("single", "missing_info", "multiple")}

def parse(text):
    text = text.split("<end_of_turn>")[0]
    starts = [i for i in (text.find("["), text.find("{")) if i >= 0]
    try:
        value = json.JSONDecoder().raw_decode(text, min(starts))[0] if starts else None
    except json.JSONDecodeError:
        return None
    return value[0] if isinstance(value, list) and len(value) == 1 else value

@torch.no_grad()
def predict(model, examples, batch_size=8):
    model.eval()
    order = sorted(examples, key=lambda ex: len(build_prompt(ex)))
    stop = [tokenizer.eos_token_id]
    eot = tokenizer.convert_tokens_to_ids("<end_of_turn>")
    if isinstance(eot, int) and eot >= 0:
        stop.append(eot)
    out = {}
    for i in range(0, len(order), batch_size):
        batch = order[i:i+batch_size]
        inputs = tokenizer([build_prompt(ex) for ex in batch], return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
        tokens = model.generate(**inputs, max_new_tokens=256, do_sample=False, eos_token_id=stop, pad_token_id=tokenizer.pad_token_id)
        for ex, seq in zip(batch, tokens[:, inputs["input_ids"].shape[1]:]):
            out[ex["id"]] = parse(tokenizer.decode(seq, skip_special_tokens=False))
        if i % 128 == 0:
            print("pred", i, "/", len(order))
    return out


In [ ]:
import gc

def free_gpu():
    for obj_name in ("model", "base"):
        if obj_name in globals():
            try:
                del globals()[obj_name]
            except Exception:
                pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print("cuda free GB", round(torch.cuda.mem_get_info()[0]/1e9, 2))

def fine_tune(examples, epochs=2, lr=LR, batch_tokens=BATCH_TOKENS, lora_r=LORA_R):
    """Starter-style LoRA on Gemma 3 270M — fits a Colab T4 (15GB)."""
    free_gpu()
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=DTYPE,
        attn_implementation="eager",
        token=HF_TOKEN,
        low_cpu_mem_usage=True,
    ).to(DEVICE)
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    lora_config = LoraConfig(task_type="CAUSAL_LM", r=lora_r, lora_alpha=32, target_modules="all-linear")
    model = get_peft_model(model, lora_config)
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()
    model.print_trainable_parameters()

    rows = []
    for ex in examples:
        prompt = tokenizer(build_prompt(ex), add_special_tokens=False)["input_ids"]
        answer = tokenizer(target_text(ex["answer"]), add_special_tokens=False)["input_ids"]
        rows.append((prompt + answer, [-100] * len(prompt) + answer))

    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    losses = []
    start = time.time()
    base = model.get_base_model()
    for ep in range(epochs):
        random.Random(SEED + ep).shuffle(rows)
        packed = sorted(rows, key=lambda r: len(r[0]) // 32)
        batches, batch = [], []
        for row in packed:
            if batch and (len(batch) + 1) * len(row[0]) > batch_tokens:
                batches.append(batch)
                batch = []
            batch.append(row)
        if batch:
            batches.append(batch)
        random.Random(SEED + ep).shuffle(batches)
        nsteps = max(1, len(batches))
        schedule = torch.optim.lr_scheduler.LambdaLR(
            optimizer, lambda s, n=nsteps: min(1, (s + 1) / 10) * 0.5 * (1 + math.cos(math.pi * s / n))
        )
        model.train()
        for bi, batch in enumerate(batches):
            width = max(len(ids) for ids, _ in batch)
            ids = torch.tensor([[tokenizer.pad_token_id] * (width - len(i)) + i for i, _ in batch], device=DEVICE)
            labels = torch.tensor([[-100] * (width - len(l)) + l for _, l in batch], device=DEVICE)
            mask = (ids != tokenizer.pad_token_id).long()
            hidden = base.model(
                input_ids=ids,
                attention_mask=mask,
                position_ids=(mask.cumsum(1) - 1).clamp(min=0),
            ).last_hidden_state
            targets, keep = labels[:, 1:], labels[:, 1:] != -100
            logits = base.lm_head(hidden[:, :-1][keep]).float()
            loss = torch.nn.functional.cross_entropy(logits, targets[keep])
            if math.isfinite(loss.item()):
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            schedule.step()
            optimizer.zero_grad(set_to_none=True)
            losses.append(loss.item())
            del ids, labels, mask, hidden, logits, loss
            if bi % 50 == 0:
                print(f"epoch {ep+1}/{epochs} step {bi}/{len(batches)} last={losses[-1]:.4f} alloc={torch.cuda.memory_allocated()/1e9:.2f}GB")
                torch.cuda.empty_cache()
    print(f"done {len(examples)} x {epochs} ep in {(time.time()-start)/60:.1f} min")
    return model, losses


In [ ]:
# If you already OOM'd, run this cell after Runtime → Restart session, from the top.
free_gpu()
model, losses = fine_tune(train_pool, epochs=EPOCHS)
val_score = score(predict(model, validation, batch_size=8))
print("validation", val_score)
if val_score["overall"] < 0.90:
    print("Val < 90% — keep 270M, train all data, 2 epochs. Do not switch to 1B/4B on a T4.")


In [ ]:
adapter_dir = Path("adapter")
adapter_dir.mkdir(exist_ok=True)
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print("saved", adapter_dir.resolve())

predictions = predict(model, dev)
empty = {"tool": "", "parameters": {}}
payload = {k: (v if v is not None else empty) for k, v in predictions.items()}
Path("submission.json").write_text(json.dumps(payload, ensure_ascii=False, indent=1))
Path("predictions.json").write_text(json.dumps(payload, ensure_ascii=False))
with zipfile.ZipFile("submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write("predictions.json")
    z.write("submission.json")
print(len(payload), "-> submission.zip  (upload this on Codabench)")


In [ ]:
GH_REPO = os.environ.get("GH_REPO", "https://github.com/maick-code/airf-slm-tool-calling-hackathon.git")
BRANCH = os.environ.get("GH_BRANCH", "arena/7d628867-airf-slm-tool-calling-hackatho")
try:
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
except Exception:
    token = os.environ.get("GH_TOKEN")
if token:
    import subprocess
    auth = GH_REPO.replace("https://", f"https://x-access-token:{token}@")
    subprocess.check_call(["git", "config", "user.email", "colab@local"])
    subprocess.check_call(["git", "config", "user.name", "colab"])
    subprocess.call(["git", "init"])
    subprocess.call(["git", "remote", "add", "origin", auth])
    subprocess.call(["git", "fetch", "origin", BRANCH])
    subprocess.call(["git", "checkout", "-B", BRANCH])
    subprocess.check_call(["git", "add", "submission.json", "submission.zip"])
    subprocess.call(["git", "commit", "-m", "Colab submission"])
    subprocess.check_call(["git", "push", "origin", BRANCH])
    print("pushed to", BRANCH)
else:
    print("No GH_TOKEN — download submission.zip from Colab files and upload to Codabench.")
